#Satellite communication market Dashboard

#Part 1 Load all of the Data

In [1]:
import pandas as pd
RAW = "data/raw/"

In [2]:
naive = pd.read_csv(RAW + "psatcat.tsv", sep="\t")
naive.head(3)

,#JCAT,Piece,Name,LDate,TLast,TOp,TDate,TF,Program,Plane,...,UNReg,UNPeriod,UNPerigee,UNApogee,UNInc,DispEpoch,DispPeri,DispApo,DispInc,Comment
0,# Updated 2026 Nov 13 0021:57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S00002,1957 ALP 2,First Sputnik,1957 Oct 4,1957 Oct 27,1957 Oct 27,1957 Oct 27,-,Sputnik,-,...,A/AC.105/INF.002,-,227,947,65.1,-,0.0,0.0,0.0,First Earth satellite
2,S00003,1957 BET 1,Second Sputnik,1957 Nov 3,1957 Nov 9,1957 Nov 9,1957 Nov 9,-,Sputnik,-,...,A/AC.105/INF.002,-,225,1671,65.3,-,0.0,0.0,0.0,First animal payload


In [3]:
naive.iloc[1, 0]

'S00002    '

In [4]:
def read_gcat(path):
    df = pd.read_csv(path, sep="\t", skiprows=[1], dtype=str, keep_default_na=False)
    df.columns = [c.lstrip("#").strip() for c in df.columns]
    df = df.apply(lambda col: col.str.strip())
    return df.replace({"-": None, "": None})

In [5]:
psatcat = read_gcat(RAW + "psatcat.tsv")
satcat = read_gcat(RAW + "satcat.tsv")
orgs = read_gcat(RAW + "orgs.tsv")

print(len(psatcat), len(satcat), len(orgs))
psatcat.head(3)

27940 69999 4145


,JCAT,Piece,Name,LDate,TLast,TOp,TDate,TF,Program,Plane,...,UNReg,UNPeriod,UNPerigee,UNApogee,UNInc,DispEpoch,DispPeri,DispApo,DispInc,Comment
0,S00002,1957 ALP 2,First Sputnik,1957 Oct 4,1957 Oct 27,1957 Oct 27,1957 Oct 27,None,Sputnik,None,...,A/AC.105/INF.002,None,227,947,65.1,None,0,0,0.00,First Earth satellite
1,S00003,1957 BET 1,Second Sputnik,1957 Nov 3,1957 Nov 9,1957 Nov 9,1957 Nov 9,None,Sputnik,None,...,A/AC.105/INF.002,None,225,1671,65.3,None,0,0,0.00,First animal payload
2,S00004,1958 ALP,Explorer 1,1958 Feb 1,1958 Apr 1,1958 Apr 1,1958 Apr 1,None,Explorer ABMA,None,...,A/AC.105/INF.001,None,None,None,None,None,0,0,0.00,Discovered trapped radiation


In [6]:
celestrak = pd.read_csv(RAW + "satcat.csv", dtype=str, keep_default_na=False)
print(len(celestrak))
celestrak.head(3)

71323


,OBJECT_NAME,OBJECT_ID,NORAD_CAT_ID,OBJECT_TYPE,OPS_STATUS_CODE,OWNER,LAUNCH_DATE,LAUNCH_SITE,DECAY_DATE,PERIOD,INCLINATION,APOGEE,PERIGEE,RCS,DATA_STATUS_CODE,ORBIT_CENTER,ORBIT_TYPE
0,SL-1 R/B,1957-001A,1,R/B,D,CIS,1957-10-04,TYMSC,1957-12-01,96.19,65.10,938,214,20.4200,,EA,IMP
1,SPUTNIK 1,1957-001B,2,PAY,D,CIS,1957-10-04,TYMSC,1958-01-03,96.10,65.00,1080,64,,,EA,IMP
2,SPUTNIK 2,1957-002A,3,PAY,D,CIS,1957-11-03,TYMSC,1958-04-14,103.74,65.33,1659,211,0.0800,,EA,IMP


In [7]:
satcat[["JCAT", "Satcat", "LDate", "DDate"]].head(3)

,JCAT,Satcat,LDate,DDate
0,S00001,00001,1957 Oct 4,1957 Dec 1 1000?
1,S00002,00002,1957 Oct 4,1958 Jan 4?
2,S00003,00003,1957 Nov 3,1958 Apr 14 0200?


In [8]:
def parse_gcat_date(col):
    day = col.str.replace("?", "", regex=False).str[:11].str.strip()
    return pd.to_datetime(day, format="%Y %b %d", errors="coerce")

In [9]:
satcat["LDate"] = parse_gcat_date(satcat["LDate"])
satcat["DDate"] = parse_gcat_date(satcat["DDate"].fillna(""))

print("launch dates that failed:", satcat["LDate"].isna().sum())
print("earliest / latest launch:", satcat["LDate"].min(), satcat["LDate"].max())
satcat[["JCAT", "LDate", "DDate"]].head(3)

launch dates that failed: 18
earliest / latest launch: 1957-10-04 00:00:00 2026-07-11 00:00:00


,JCAT,LDate,DDate
0,S00001,1957-10-04,1957-12-01
1,S00002,1957-10-04,1958-01-04
2,S00003,1957-11-03,1958-04-14


In [10]:
satcat["Satcat"] = pd.to_numeric(satcat["Satcat"], errors="coerce").astype("Int64")
celestrak["NORAD_CAT_ID"] = pd.to_numeric(celestrak["NORAD_CAT_ID"]).astype("Int64")

print("GCAT objects without a catalogue number:", satcat["Satcat"].isna().sum())
satcat[["JCAT", "Satcat"]].head(3)

GCAT objects without a catalogue number: 563


,JCAT,Satcat
0,S00001,1
1,S00002,2
2,S00003,3


In [11]:
for name, df in [("psatcat", psatcat), ("satcat", satcat), ("orgs", orgs), ("celestrak", celestrak)]:
    print(f"{name}: {len(df):,} rows, {len(df.columns)} columns")

psatcat: 27,940 rows, 28 columns
satcat: 69,999 rows, 42 columns
orgs: 4,145 rows, 17 columns
celestrak: 71,323 rows, 17 columns


#Part 2 

In [12]:
com = psatcat[psatcat["Category"].str.startswith("COM")]
print(len(com))
com["Category"].value_counts().head(8)

17750


Category
COM           17443
COM/MET-RO      215
COM?             37
COM*             15
COM/TECH         14
COM/MET          12
COM/IMG           5
COM/SCI           5
Name: count, dtype: int64

In [13]:
com["Class"].value_counts()

Class
B     15754
D      1170
C       659
A       163
CB        2
CD        2
Name: count, dtype: int64

In [14]:
sats = com[["JCAT", "Category", "Class"]].merge(satcat, on="JCAT", how="inner")
print(len(sats))
sats[["JCAT", "Name", "LDate", "Owner", "State", "Status"]].head(3)

17750


,JCAT,Name,LDate,Owner,State,Status
0,S00010,SCORE,1958-12-18,ARPA/SCEL,US,AR
1,S00049,Echo 1,1960-08-12,LARCN,US,R
2,S00058,Courier 1B,1960-10-04,SCEL,US,O


In [15]:
sats = com[["JCAT", "Category", "Class"]].merge(satcat, on="JCAT", how="inner")
print(len(sats))
sats[["JCAT", "Name", "LDate", "Owner", "State", "Status"]].head(3)

17750


,JCAT,Name,LDate,Owner,State,Status
0,S00010,SCORE,1958-12-18,ARPA/SCEL,US,AR
1,S00049,Echo 1,1960-08-12,LARCN,US,R
2,S00058,Courier 1B,1960-10-04,SCEL,US,O


In [16]:
missing = ~sats["Owner"].isin(orgs["Code"])
print("owners not found in orgs:", missing.sum())
sats.loc[missing, "Owner"].value_counts().head(5)

owners not found in orgs: 350


Owner
NPOPM/MSVYAZ    20
SPIREL/SPIRE    15
UNKS/MSVYAZ     11
SAMSO/DCA       10
NRO/SAFSP       10
Name: count, dtype: int64

In [17]:
sats["owner_code"] = sats["Owner"].str.split("/").str[0]

print("owners not found now:", (~sats["owner_code"].isin(orgs["Code"])).sum())
print("different owners:", sats["owner_code"].nunique())
sats["owner_code"].value_counts().head(5)

owners not found now: 0
different owners: 521


owner_code
SPXS       12446
ONEWEBN      526
KUIP         398
GUKOSR       245
YUANX        239
Name: count, dtype: int64

In [18]:
sats = sats.merge(celestrak[["NORAD_CAT_ID", "OPS_STATUS_CODE"]],
                  left_on="Satcat", right_on="NORAD_CAT_ID", how="left")
print(len(sats))
print("not found in CelesTrak:", sats["NORAD_CAT_ID"].isna().sum())
sats["OPS_STATUS_CODE"].value_counts()

17750
not found in CelesTrak: 86


OPS_STATUS_CODE
+    13019
D     2790
-      822
       695
P      333
B        5
Name: count, dtype: int64

-350 satellites have joint owners ("NPOPM/MSVYAZ") -> kept the lead (first) owner, so the design stays 1 owner per satellite
-86 satellites not in CelesTrak (no catalogue number); 695 in CelesTrak but with no status

In [19]:
import sqlite3

con = sqlite3.connect("satcom.db")
con.execute("PRAGMA foreign_keys = ON")

In [20]:
schema = """
DROP TABLE IF EXISTS satellite;
DROP TABLE IF EXISTS organisation;
DROP TABLE IF EXISTS country;

CREATE TABLE country (
    code        TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    esa_member  INTEGER NOT NULL CHECK (esa_member IN (0, 1))
);

CREATE TABLE organisation (
    code          TEXT PRIMARY KEY,
    name          TEXT NOT NULL,
    country_code  TEXT NOT NULL REFERENCES country(code)
);

CREATE TABLE satellite (
    jcat         TEXT PRIMARY KEY,
    name         TEXT NOT NULL,
    launch_date  TEXT NOT NULL,
    decay_date   TEXT,
    status       TEXT NOT NULL,
    category     TEXT NOT NULL,
    class        TEXT,
    owner_code   TEXT NOT NULL REFERENCES organisation(code),
    norad        INTEGER UNIQUE,
    ops_status   TEXT
);

CREATE INDEX idx_satellite_owner  ON satellite(owner_code);
CREATE INDEX idx_satellite_launch ON satellite(launch_date);
"""
con.executescript(schema)

In [21]:
pd.read_sql("SELECT name, type FROM sqlite_master ORDER BY type, name", con)

,name,type
0,idx_satellite_launch,index
1,idx_satellite_owner,index
2,sqlite_autoindex_country_1,index
3,sqlite_autoindex_organisation_1,index
4,sqlite_autoindex_satellite_1,index
5,sqlite_autoindex_satellite_2,index
6,country,table
7,organisation,table
8,satellite,table


In [22]:
try:
    con.execute("INSERT INTO organisation VALUES ('TEST', 'Test org', 'XX')")
except sqlite3.IntegrityError as e:
    print("With PRAGMA -> refused:", e)
con.rollback()

con2 = sqlite3.connect("satcom.db")
con2.execute("INSERT INTO organisation VALUES ('TEST', 'Test org', 'XX')")
print("Without PRAGMA -> accepted! (bad data got in)")
con2.rollback()
con2.close()

With PRAGMA -> refused: FOREIGN KEY constraint failed
Without PRAGMA -> accepted! (bad data got in)


In [23]:
ESA = {"AT", "B", "CZ", "DK", "EE", "FI", "F", "D", "GR", "HU", "IE", "I", "L",
       "NL", "N", "PL", "P", "RO", "SI", "E", "S", "CH", "UK"}

org = orgs[orgs["Code"].isin(sats["owner_code"])]
ctry = orgs[orgs["Code"].isin(org["StateCode"])]

country = ctry.assign(name=ctry["ShortEName"].fillna(ctry["EName"]),
                      esa_member=ctry["Code"].isin(ESA).astype(int))
country = country.rename(columns={"Code": "code"})[["code", "name", "esa_member"]]

print(len(country), "countries,", country["esa_member"].sum(), "of them ESA members")
country[country["esa_member"] == 1].head()

79 countries, 19 of them ESA members


,code,name,esa_member
15,AT,Austria,1
18,B,Belgium,1
37,CH,Switzerland,1
71,CZ,Czech Republic,1
72,D,Germany,1


In [24]:
organisation = org.assign(name=org["EName"].fillna(org["Name"]))
organisation = organisation.rename(columns={"Code": "code", "StateCode": "country_code"})
organisation = organisation[["code", "name", "country_code"]]

print(len(organisation), "organisations")
organisation[organisation["code"].isin(["SPXS", "ONEWEBN", "KUIP"])]

521 organisations


,code,name,country_code
2039,KUIP,Kuiper Systems LLC,US
2740,ONEWEBN,One Web (Network Access Associates Ltd.),UK
3385,SPXS,SpaceX (Seattle),US


In [25]:
satellite = sats.rename(columns={"JCAT": "jcat", "Name": "name", "Status": "status",
                                 "Category": "category", "Class": "class",
                                 "Satcat": "norad", "OPS_STATUS_CODE": "ops_status"})
satellite["launch_date"] = sats["LDate"].dt.strftime("%Y-%m-%d")
satellite["decay_date"] = sats["DDate"].dt.strftime("%Y-%m-%d")
satellite["ops_status"] = satellite["ops_status"].replace("", None)
satellite = satellite[["jcat", "name", "launch_date", "decay_date", "status", "category",
                       "class", "owner_code", "norad", "ops_status"]]

print(len(satellite), "satellites")
satellite.head(3)

17750 satellites


,jcat,name,launch_date,decay_date,status,category,class,owner_code,norad,ops_status
0,S00010,SCORE,1958-12-18,1959-01-21,AR,COM,D,ARPA,10,D
1,S00049,Echo 1,1960-08-12,1968-05-24,R,COM,C,LARCN,49,D
2,S00058,Courier 1B,1960-10-04,NaN,O,COM,D,SCEL,58,NaN


In [26]:
satellite = sats.rename(columns={"JCAT": "jcat", "Name": "name", "Status": "status",
                                 "Category": "category", "Class": "class",
                                 "Satcat": "norad", "OPS_STATUS_CODE": "ops_status"})
satellite["launch_date"] = sats["LDate"].dt.strftime("%Y-%m-%d")
satellite["decay_date"] = sats["DDate"].dt.strftime("%Y-%m-%d")
satellite["ops_status"] = satellite["ops_status"].replace("", None)
satellite = satellite[["jcat", "name", "launch_date", "decay_date", "status", "category",
                       "class", "owner_code", "norad", "ops_status"]]

print(len(satellite), "satellites")
satellite.head(3)

17750 satellites


,jcat,name,launch_date,decay_date,status,category,class,owner_code,norad,ops_status
0,S00010,SCORE,1958-12-18,1959-01-21,AR,COM,D,ARPA,10,D
1,S00049,Echo 1,1960-08-12,1968-05-24,R,COM,C,LARCN,49,D
2,S00058,Courier 1B,1960-10-04,NaN,O,COM,D,SCEL,58,NaN


In [27]:
for name, df in [("country", country), ("organisation", organisation), ("satellite", satellite)]:
    df.to_sql(name, con, if_exists="append", index=False)
con.commit()
print("loaded")

loaded


In [28]:
q = lambda sql: con.execute(sql).fetchall()

print("satellites:", q("SELECT COUNT(*) FROM satellite")[0][0])
print("without a known owner:", q("""
    SELECT COUNT(*) FROM satellite s
    LEFT JOIN organisation o ON o.code = s.owner_code
    WHERE o.code IS NULL""")[0][0])
print("no CelesTrak status:", q("SELECT COUNT(*) FROM satellite WHERE ops_status IS NULL")[0][0])
print("foreign key problems:", q("PRAGMA foreign_key_check"))

pd.read_sql("""
    SELECT o.name AS owner, c.name AS country, COUNT(*) AS satellites
    FROM satellite s
    JOIN organisation o ON o.code = s.owner_code
    JOIN country c ON c.code = o.country_code
    GROUP BY o.code
    ORDER BY satellites DESC
    LIMIT 5""", con)

satellites: 17750
without a known owner: 0
no CelesTrak status: 781
foreign key problems: []


,owner,country,satellites
0,SpaceX (Seattle),USA,12446
1,One Web (Network Access Associates Ltd.),UK,526
2,Kuiper Systems LLC,USA,398
3,Main Space Forces Directorate,USSR,245
4,Shanghai Spacecom Satellite Technology Ltd,China,239


In [29]:
pip install nbformat


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import plotly.express as px

ACTIVE = "s.ops_status IN ('+', 'P', 'B', 'S', 'X')"

In [32]:
growth = pd.read_sql("""
    SELECT CAST(substr(s.launch_date, 1, 4) AS INTEGER) AS year,
           o.name AS owner,
           COUNT(*) AS launched
    FROM satellite s
    JOIN organisation o ON o.code = s.owner_code
    GROUP BY year, o.code
    HAVING COUNT(*) >= 10
    ORDER BY year""", con)

fig = px.bar(growth, x="year", y="launched", color="owner", template="simple_white")
fig.show(config={"displayModeBar": False})

In [33]:
per_year = pd.read_sql("""
    SELECT CAST(substr(launch_date, 1, 4) AS INTEGER) AS year, COUNT(*) AS launched
    FROM satellite
    GROUP BY year""", con).set_index("year")["launched"]

print("2015:", per_year[2015], "| 2025:", per_year[2025])

2015: 68 | 2025: 3745


In [34]:
by_owner = pd.read_sql(f"""
    SELECT o.name AS owner,
           COUNT(*) AS active,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM satellite s WHERE {ACTIVE}), 1) AS share_pct
    FROM satellite s
    JOIN organisation o ON o.code = s.owner_code
    WHERE {ACTIVE}
    GROUP BY o.code
    ORDER BY active DESC
    LIMIT 10""", con)

fig = px.bar(by_owner, x="share_pct", y="owner", orientation="h", template="simple_white",
             color_discrete_sequence=["grey"])
fig.update_yaxes(autorange="reversed")
fig.show(config={"displayModeBar": False})

In [35]:
by_country = pd.read_sql(f"""
    SELECT c.name AS country,
           COUNT(*) AS active,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM satellite s WHERE {ACTIVE}), 1) AS share_pct
    FROM satellite s
    JOIN organisation o ON o.code = s.owner_code
    JOIN country c ON c.code = o.country_code
    WHERE {ACTIVE}
    GROUP BY c.code
    ORDER BY active DESC
    LIMIT 10""", con)
by_country

,country,active,share_pct
0,USA,11450,85.7
1,UK,725,5.4
2,China,569,4.3
3,Russia,177,1.3
4,France,52,0.4
5,Luxembourg,32,0.2
6,Turkey,28,0.2
7,Canada,28,0.2
8,Japan,25,0.2
9,Australia,25,0.2


In [36]:
europe = pd.read_sql("""
    WITH RECURSIVE years(y) AS (
        SELECT 2016
        UNION ALL
        SELECT y + 1 FROM years WHERE y < 2025
    )
    SELECT y AS year,
           COUNT(*) AS in_orbit,
           SUM(c.esa_member) AS esa,
           ROUND(100.0 * SUM(c.esa_member) / COUNT(*), 1) AS esa_share_pct
    FROM years
    JOIN satellite s ON s.launch_date <= y || '-12-31'
                    AND (s.decay_date IS NULL OR s.decay_date > y || '-12-31')
    JOIN organisation o ON o.code = s.owner_code
    JOIN country c ON c.code = o.country_code
    GROUP BY y
    ORDER BY y""", con)

display(europe)
fig = px.line(europe, x="year", y="esa_share_pct", markers=True, template="simple_white",
              color_discrete_sequence=["grey"])
fig.show(config={"displayModeBar": False})

,year,in_orbit,esa,esa_share_pct
0,2016,1875,134,7.1
1,2017,1970,147,7.5
2,2018,2053,174,8.5
3,2019,2250,197,8.8
4,2020,3210,307,9.6
5,2021,4564,621,13.6
6,2022,6382,758,11.9
7,2023,8444,916,10.8
8,2024,10091,939,9.3
9,2025,13024,922,7.1


In [37]:
pd.read_sql(f"""
    SELECT CASE substr(s.class, 1, 1)
               WHEN 'B' THEN 'business' WHEN 'D' THEN 'defence'
               WHEN 'C' THEN 'civil'    WHEN 'A' THEN 'amateur'
               ELSE 'other' END AS use,
           COUNT(*) AS active
    FROM satellite s
    WHERE {ACTIVE}
    GROUP BY use
    ORDER BY active DESC""", con)

,use,active
0,business,12763
1,civil,316
2,defence,237
3,amateur,41
